In [16]:
import warnings
#warnings.simplefilter(action='ignore')
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from pycaret.utils import version
version()
import scipy

# Compatibilità temporanea con librerie che importano scipy.interp
if not hasattr(scipy, "interp"):
    scipy.interp = np.interp

## Best models

In [17]:
import pandas as pd
fold_col = "fold"           # Nome della colonna con fold_1, ..., fold_5
id_col = "patient_id"

df_train = pd.read_csv("folds_icd/train_STATIC+BiPadLSTM_0.csv")
df_test = pd.read_csv("folds_icd/test_STATIC+BiPadLSTM_0.csv")
assert set(df_train[fold_col]) == set(df_test[fold_col])

In [18]:
import pandas as pd
from pycaret.classification import ClassificationExperiment
from sklearn.metrics import brier_score_loss

fold_col = "fold"
id_col = "patient_id"
target = "target"

if df_train[fold_col].isna().any() or df_test[fold_col].isna().any():
    raise ValueError("Sono presenti fold mancanti.")

if set(df_train[fold_col]) != set(df_test[fold_col]):
    raise ValueError("I fold del train e del test non coincidono.")

from sklearn.metrics import confusion_matrix, precision_recall_curve, auc
from IPython.display import display

# Classificazione binaria: 0 = negativo, 1 = positivo.
def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    # NPV non definito se non ci sono predizioni negative.
    return tn / (tn + fn) if (tn + fn) > 0 else float("nan")


def auprc_score(y_true, y_score):
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    return auc(recall, precision)

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    return tn / (tn + fp) if (tn + fp) > 0 else float("nan")

risultati = []

for fold_id in df_train[fold_col].unique():
    print(f"Elaborazione fold: {fold_id}", flush=True)

    train_fold = df_train.loc[
        df_train[fold_col] == fold_id
    ].copy()

    test_fold = df_test.loc[
        df_test[fold_col] == fold_id
    ].copy()

    # Controlla che i pazienti del test non siano nel train.
    if train_fold[id_col].isna().any() or test_fold[id_col].isna().any():
        raise ValueError(f"Patient ID mancanti nel fold {fold_id}.")

    overlap = set(train_fold[id_col]) & set(test_fold[id_col])
    if overlap:
        raise ValueError(
            f"Fold {fold_id}: {len(overlap)} pazienti "
            "presenti sia nel train sia nel test."
        )

    # Fold e identificativo non devono essere feature.
    train_fold = train_fold.drop(
        columns=[fold_col, id_col]
    ).reset_index(drop=True)

    test_fold = test_fold.drop(
        columns=[fold_col, id_col]
    ).reset_index(drop=True)

    exp = ClassificationExperiment()

    exp.setup(
        data=train_fold,
        target=target,
        test_data=test_fold,
        index=False,
        session_id=123,
        use_gpu=False,
        fix_imbalance=True,
        verbose=False,
        html=False,
    )

    exp.add_metric(
        id="brier",
        name="Brier Score",
        score_func=brier_score_loss,
        target="pred_proba",
        greater_is_better=False,
    )

    exp.add_metric(
        id="npv",
        name="NPV",
        score_func=npv_score,
        target="pred",
        greater_is_better=True,
    )
    exp.add_metric(
        id="auprc",
        name="AUPRC",
        score_func=auprc_score,
        target="pred_proba",
        greater_is_better=True,
    )
    exp.compare_models(
        cross_validation=False,  # Valuta sul test_fold fornito
        sort="MCC",
        round=3,
        errors="raise",
        verbose=False,
    )
    exp.add_metric(
        id="specificity",
        name="specificity",
        score_func=specificity_score,
        target="pred",
        greater_is_better=True,
    )    
    tabella = exp.pull().copy()
    tabella["fold"] = fold_id
    risultati.append(tabella)

    print(f"Risultati fold {fold_id}")
    with pd.option_context("display.max_columns", None):
        display(tabella)
    
    tabella = exp.pull().copy()
    tabella["fold"] = fold_id
    risultati.append(tabella)

    print(f"Fold {fold_id} completato.", flush=True)

# Risultati di tutti i modelli in tutti i fold.
risultati_fold = pd.concat(risultati, ignore_index=True)
risultati_fold

Elaborazione fold: 1
Risultati fold 1


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,Brier Score,NPV,AUPRC,TT (Sec),fold
xgboost,Extreme Gradient Boosting,0.933,0.862,0.68,0.630,0.654,0.617,0.617,0.067,0.967,0.625,3.46,1
gbc,Gradient Boosting Classifier,0.929,0.915,0.68,0.607,0.642,0.602,0.604,0.063,0.967,0.633,31.78,1
lightgbm,Light Gradient Boosting Machine,0.925,0.905,0.68,0.586,0.630,0.588,0.590,0.069,0.966,0.618,0.75,1
ada,Ada Boost Classifier,0.922,0.908,0.68,0.567,0.618,0.575,0.578,0.131,0.966,0.562,6.01,1
knn,K Neighbors Classifier,0.899,0.845,0.72,0.474,0.571,0.517,0.532,0.075,0.970,0.603,0.06,1
et,Extra Trees Classifier,0.896,0.872,0.68,0.460,0.548,0.492,0.504,0.080,0.965,0.461,0.15,1
lr,Logistic Regression,0.869,0.891,0.76,0.396,0.520,0.454,0.486,0.092,0.973,0.520,0.07,1
rf,Random Forest Classifier,0.884,0.858,0.64,0.421,0.508,0.446,0.458,0.085,0.961,0.470,0.23,1
svm,SVM - Linear Kernel,0.866,0.782,0.68,0.378,0.486,0.416,0.439,0.134,0.964,0.544,0.09,1
dt,Decision Tree Classifier,0.896,0.727,0.52,0.448,0.482,0.424,0.425,0.104,0.950,0.506,1.04,1


Fold 1 completato.
Elaborazione fold: 2


KeyboardInterrupt: 

In [ ]:
risultati_fold = pd.concat(risultati, ignore_index=True)

metriche = [
    col
    for col in risultati_fold.select_dtypes(include="number").columns
    if col not in ["fold", "TT (Sec)"]
]

statistiche = (
    risultati_fold
    .groupby("Model")[metriche]
    .agg(["mean", "std"])
    .sort_values(("MCC", "mean"), ascending=False)
)

riepilogo = pd.DataFrame(index=statistiche.index)

for metrica in metriche:
    riepilogo[metrica] = [
        f"{media:.3f}±{std:.3f}" if pd.notna(std)
        else f"{media:.3f}±NA"
        for media, std in zip(
            statistiche[(metrica, "mean")],
            statistiche[(metrica, "std")],
        )
    ]

riepilogo = riepilogo.rename(columns={
    "Prec.": "PPV",
    "Recall": "Sensitivity",
    "Brier Score": "Brier"
}).drop(columns=["Kappa"])
ncols = ["AUC", "AUPRC", "F1", "PPV", "Sensitivity", "NPV", "MCC", "Accuracy", "Brier"]
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(riepilogo[ncols])

,AUC,AUPRC,F1,PPV,Sensitivity,NPV,MCC,Accuracy,Brier
Model,,,,,,,,,
Gradient Boosting Classifier,0.911±0.035,0.701±0.083,0.691±0.058,0.647±0.060,0.752±0.120,0.973±0.013,0.661±0.068,0.936±0.011,0.054±0.007
Extreme Gradient Boosting,0.921±0.035,0.715±0.104,0.666±0.066,0.697±0.041,0.643±0.104,0.962±0.010,0.635±0.070,0.939±0.010,0.049±0.010
Ada Boost Classifier,0.908±0.033,0.624±0.059,0.662±0.059,0.583±0.060,0.775±0.113,0.975±0.013,0.631±0.070,0.924±0.014,0.215±0.004
Random Forest Classifier,0.917±0.031,0.683±0.098,0.637±0.074,0.763±0.067,0.550±0.084,0.954±0.009,0.617±0.076,0.940±0.011,0.049±0.007
Light Gradient Boosting Machine,0.922±0.027,0.718±0.085,0.638±0.071,0.701±0.038,0.588±0.104,0.957±0.010,0.607±0.073,0.937±0.010,0.050±0.009
Extra Trees Classifier,0.894±0.032,0.638±0.073,0.607±0.038,0.765±0.038,0.504±0.041,0.949±0.004,0.589±0.039,0.937±0.005,0.052±0.005
Decision Tree Classifier,0.758±0.035,0.534±0.045,0.532±0.044,0.585±0.061,0.496±0.071,0.947±0.007,0.492±0.044,0.916±0.010,0.080±0.009
SVM - Linear Kernel,0.815±0.064,0.587±0.076,0.518±0.063,0.397±0.046,0.752±0.130,0.971±0.015,0.481±0.083,0.867±0.016,0.133±0.016
K Neighbors Classifier,0.859±0.053,0.558±0.064,0.505±0.081,0.383±0.057,0.744±0.147,0.970±0.017,0.467±0.102,0.861±0.023,0.100±0.014


In [ ]:
def create_table_with_splits(formatted_results, nsplit=4):
    """Split columns into balanced blocks and stack them vertically."""
    if not isinstance(nsplit, int) or nsplit < 1:
        raise ValueError("nsplit must be a positive integer.")
    if formatted_results.shape[1] == 0:
        raise ValueError("The DataFrame must have at least one column.")

    # Avoid empty blocks
    n = len(formatted_results.columns)
    nsplit = min(nsplit, n)

    base, remainder = divmod(n, nsplit)
    sizes = [base + (i < remainder) for i in range(nsplit)]

    blocks = []
    start = 0
    for size in sizes:
        blocks.append(
            formatted_results.iloc[:, start:start + size].copy()
        )
        start += size

    width = max(sizes)
    final_columns = ["Model"] + blocks[0].columns.tolist()

    parts = []
    for i, block in enumerate(blocks):
        if i > 0:
            header = (
                ["Model"]
                + block.columns.tolist()
                + [""] * (width - block.shape[1])
            )
            parts.append(
                pd.DataFrame([header], columns=final_columns)
            )

        rows = block.rename_axis("Model").reset_index()
        rows.columns = final_columns[:rows.shape[1]]
        rows = rows.reindex(columns=final_columns, fill_value="")
        parts.append(rows)

    return pd.concat(parts, ignore_index=True)

In [ ]:
from scripts.utils import highlight_best_values
table = create_table_with_splits(riepilogo, nsplit=2)
table.set_index("Model", inplace=True)
print(highlight_best_values(table.to_latex()))

\begin{tabular}{llllll}
\toprule
 & Accuracy & AUC & Sensitivity & PPV & F1 \\
Model &  &  &  &  &  \\
\midrule
Gradient Boosting Classifier & 0.936±0.011 & 0.911±0.035 & 0.752±0.120 & 0.647±0.060 & \textbf{0.691}±0.058 \\
Extreme Gradient Boosting & 0.939±0.010 & 0.921±0.035 & 0.643±0.104 & 0.697±0.041 & 0.666±0.066 \\
Ada Boost Classifier & 0.924±0.014 & 0.908±0.033 & 0.775±0.113 & 0.583±0.060 & 0.662±0.059 \\
Random Forest Classifier & \textbf{0.940}±0.011 & 0.917±0.031 & 0.550±0.084 & 0.763±0.067 & 0.637±0.074 \\
Light Gradient Boosting Machine & 0.937±0.010 & \textbf{0.922}±0.027 & 0.588±0.104 & 0.701±0.038 & 0.638±0.071 \\
Extra Trees Classifier & 0.937±0.005 & 0.894±0.032 & 0.504±0.041 & \textbf{0.765}±0.038 & 0.607±0.038 \\
Decision Tree Classifier & 0.916±0.010 & 0.758±0.035 & 0.496±0.071 & 0.585±0.061 & 0.532±0.044 \\
SVM - Linear Kernel & 0.867±0.016 & 0.815±0.064 & 0.752±0.130 & 0.397±0.046 & 0.518±0.063 \\
K Neighbors Classifier & 0.861±0.023 & 0.859±0.053 & 0.744±0.147 & 